In [1]:
from bs4 import BeautifulSoup as BS
import requests
import pandas as pd
from urllib.parse import urljoin


# url of the website to scrape
url ='https://books.toscrape.com/'


# send a GET request to the website
response = requests.get(url, timeout=15)
# set the response encoding to UTF-8
response.encoding = 'utf-8'
# parse the HTML content of the page with BeautifulSoup
soup = BS(response.text, 'html.parser')

In [2]:
# find the first product on the page and extract its name
product = soup.find('article', class_='product_pod')
# extract the product name from the HTML element
product_name = product.find('h3').find('a')['title']
# extract the product price from the HTML element
product_price = product.find('p', class_='price_color').get_text(strip=True)
# convert the product price to a float after removing the currency symbol
product_price = float(product_price.replace('£', ''))
# extract the product availability from the HTML element
product_availability = product.find('p', class_ = 'availability').get_text(strip=True)
# extract the product rating from the HTML element
product_rating = product.find('p', class_='star-rating')
# get the textual representation of the product rating (e.g., 'One', 'Two', etc.)
rating_text = product_rating.get('class')[1]
# map the textual rating to a numerical value
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
# get the numerical rating from the textual representation
rating_number = rating_map[rating_text]
# find all products on the page and store them in a list
products = soup.find_all('article', class_='product_pod')

In [3]:
product_list = []
# iterate over all products and print their names and prices
for product in products:
    product_name = product.find('h3').find('a').get('title')
    product_price = product.find('p', class_='price_color').get_text(strip=True)
    product_price = float(product_price.replace('£', ''))
    product_availability = product.find('p', class_ = 'availability').get_text(strip=True)
    product_rating = product.find('p', class_='star-rating')
    rating_text = product_rating.get('class')[1]
    rating_number = rating_map[rating_text]
    # adds the name, the price, the availability, and the rating to the empty list
    product_list.append({'name': product_name, 
                        'price': product_price,
                        'availability': product_availability,
                        'rating': rating_number})
# function to extract product information from a list of products

In [4]:
def extract_products(products):
    product_list = []
    for product in products:
        product_name = product.find('h3').find('a').get('title')
        product_price = product.find('p', class_='price_color').get_text(strip=True)
        product_price = float(product_price.replace('£', ''))
        product_availability = product.find('p', class_ = 'availability').get_text(strip=True)
        product_rating = product.find('p', class_='star-rating')
        rating_text = product_rating.get('class')[1]
        rating_number = rating_map[rating_text]
        # adds the name, the price, the availability, and the rating to the empty list
        product_list.append({'name': product_name, 
                            'price': product_price,
                            'availability': product_availability,
                            'rating': rating_number,
        })
    return product_list

In [5]:
# extract product information from the second page
extract_product = extract_products(products)

df_products = pd.DataFrame(extract_product)

next_page = soup.find('li', class_='next')
next_page_url = next_page.find('a').get('href')
full_url = urljoin(url, next_page_url)

response_page2 = requests.get(full_url, timeout=15)

soup_page2 = BS(response_page2.content, 'html.parser')

product_page2 = soup_page2.find_all('article', class_='product_pod')

extracted_page2 = extract_products(product_page2)

df_page2 = pd.DataFrame(extracted_page2)

df_all_products = pd.concat(
    [df_products, df_page2],
    ignore_index=True
)


In [6]:
current_url = "https://books.toscrape.com/"
page_number = 1
all_products = []

while page_number:
    response = requests.get(current_url, timeout=15)
    response.raise_for_status()

    soup = BS(response.content, 'html.parser')

    products =soup.find_all(
        'article', class_= 'product_pod'
    )

    extracted_products = extract_products(products)
    all_products.extend(extracted_products)

    print(f"Page {page_number}: {len(extracted_products)} products")

    next_page = soup.find('li', class_='next')

    if next_page is None:
        break

    next_page_url = next_page.find('a').get('href')
    current_url = urljoin(current_url, next_page_url)

    page_number += 1

df_all_products = pd.DataFrame(all_products)

print(f"Total pages: {page_number}")
print(f"Total products: {len(df_all_products)}")
print(df_all_products.shape)

file_name = 'books_catalog.xlsx'

df_all_products.to_excel(file_name, index=False)

if __name__ == "__main__":      # só roda se você executar ESTE arquivo
    
    print(f'File saved as {file_name}')

Page 1: 20 products
Page 2: 20 products
Page 3: 20 products
Page 4: 20 products
Page 5: 20 products
Page 6: 20 products
Page 7: 20 products
Page 8: 20 products
Page 9: 20 products
Page 10: 20 products
Page 11: 20 products
Page 12: 20 products
Page 13: 20 products
Page 14: 20 products
Page 15: 20 products
Page 16: 20 products
Page 17: 20 products
Page 18: 20 products
Page 19: 20 products
Page 20: 20 products
Page 21: 20 products
Page 22: 20 products
Page 23: 20 products
Page 24: 20 products
Page 25: 20 products
Page 26: 20 products
Page 27: 20 products
Page 28: 20 products
Page 29: 20 products
Page 30: 20 products
Page 31: 20 products
Page 32: 20 products
Page 33: 20 products
Page 34: 20 products
Page 35: 20 products
Page 36: 20 products
Page 37: 20 products
Page 38: 20 products
Page 39: 20 products
Page 40: 20 products
Page 41: 20 products
Page 42: 20 products
Page 43: 20 products
Page 44: 20 products
Page 45: 20 products
Page 46: 20 products
Page 47: 20 products
Page 48: 20 products
P

In [7]:
duplicate_count = df_all_products.duplicated().sum()

print(f'Duplicate records: {duplicate_count}')

Duplicate records: 0


In [8]:
missing_value = df_all_products.isna().sum()
display(missing_value)

name            0
price           0
availability    0
rating          0
dtype: int64

In [9]:
display(df_all_products.head())

,name,price,availability,rating
0,A Light in the Attic,51.77,In stock,3
1,Tipping the Velvet,53.74,In stock,1
2,Soumission,50.10,In stock,1
3,Sharp Objects,47.82,In stock,4
4,Sapiens: A Brief History of Humankind,54.23,In stock,5


In [10]:
text_columns = df_all_products.select_dtypes(include='str')

empty_value = text_columns.apply(
    lambda column: column.str.strip().eq('').sum()

)
display(empty_value.to_frame(name='empty_count'))

,empty_count
name,0
availability,0


In [11]:
price_analysis = df_all_products['price'].describe()

display(price_analysis.to_frame(name='price_analysis'))

,price_analysis
count,1000.00000
mean,35.07035
std,14.44669
min,10.00000
25%,22.10750
50%,35.98000
75%,47.45750
max,59.99000


In [12]:

most_expensive = df_all_products.sort_values(
    by="price",
    ascending=False
)

display(most_expensive.head(10))


,name,price,availability,rating
648,The Perfect Play (Play by Play #1),59.99,In stock,3
617,Last One Home (New Beginnings #1),59.98,In stock,3
860,Civilization and Its Discontents,59.95,In stock,2
560,The Barefoot Contessa Cookbook,59.92,In stock,5
366,The Diary of a Young Girl,59.90,In stock,3
657,The Bone Hunters (Lexy Vaughan & Steven Macaul...,59.71,In stock,3
133,Thomas Jefferson and the Tripoli Pirates: The ...,59.64,In stock,1
387,Boar Island (Anna Pigeon #19),59.48,In stock,3
549,The Man Who Mistook His Wife for a Hat and Oth...,59.45,In stock,4
393,The Improbability of Love,59.45,In stock,1


In [13]:
cheapest_book = df_all_products.sort_values(
    by="price",
    ascending=True
)

display(cheapest_book.head(10))

,name,price,availability,rating
638,An Abundance of Katherines,10.00,In stock,5
501,The Origin of Species,10.01,In stock,4
716,The Tipping Point: How Little Things Can Make ...,10.02,In stock,2
84,Patience,10.16,In stock,3
302,Greek Mythic History,10.23,In stock,5
558,The Fellowship of the Ring (The Lord of the Ri...,10.27,In stock,2
479,History of Beauty,10.29,In stock,4
242,The Lucifer Effect: Understanding How Good Peo...,10.40,In stock,1
434,"NaNo What Now? Finding your editing process, r...",10.41,In stock,4
274,Pet Sematary,10.56,In stock,3


In [14]:
cheapest_book2 = df_all_products.nsmallest(10, 'price')

display(cheapest_book2)

,name,price,availability,rating
638,An Abundance of Katherines,10.00,In stock,5
501,The Origin of Species,10.01,In stock,4
716,The Tipping Point: How Little Things Can Make ...,10.02,In stock,2
84,Patience,10.16,In stock,3
302,Greek Mythic History,10.23,In stock,5
558,The Fellowship of the Ring (The Lord of the Ri...,10.27,In stock,2
479,History of Beauty,10.29,In stock,4
242,The Lucifer Effect: Understanding How Good Peo...,10.40,In stock,1
434,"NaNo What Now? Finding your editing process, r...",10.41,In stock,4
274,Pet Sematary,10.56,In stock,3


In [15]:
most_expensive_book = df_all_products.nlargest(10, 'price')

display(most_expensive_book)

,name,price,availability,rating
648,The Perfect Play (Play by Play #1),59.99,In stock,3
617,Last One Home (New Beginnings #1),59.98,In stock,3
860,Civilization and Its Discontents,59.95,In stock,2
560,The Barefoot Contessa Cookbook,59.92,In stock,5
366,The Diary of a Young Girl,59.90,In stock,3
657,The Bone Hunters (Lexy Vaughan & Steven Macaul...,59.71,In stock,3
133,Thomas Jefferson and the Tripoli Pirates: The ...,59.64,In stock,1
387,Boar Island (Anna Pigeon #19),59.48,In stock,3
393,The Improbability of Love,59.45,In stock,1
549,The Man Who Mistook His Wife for a Hat and Oth...,59.45,In stock,4


In [17]:
boosk_price_range = df_all_products [
    df_all_products['price'].between(10, 40)
]

display(boosk_price_range.head(10))
print('Total de Livros encontr5ado;', len(boosk_price_range) )

,name,price,availability,rating
5,The Requiem Red,22.65,In stock,1
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,4
7,The Coming Woman: A Novel Based on the Life of...,17.93,In stock,3
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,4
10,"Starving Hearts (Triangular Trade Trilogy, #1)",13.99,In stock,2
11,Shakespeare's Sonnets,20.66,In stock,4
12,Set Me Free,17.46,In stock,5
14,Rip it Up and Start Again,35.02,In stock,5
16,Olio,23.88,In stock,1
17,Mesaerion: The Best Science Fiction Stories 18...,37.59,In stock,1


Total de Livros encontr5ado; 597


In [19]:
boosk_price_range = df_all_products [
    df_all_products['price'].between(20, 40)
]

display(boosk_price_range.head(10))
print('Total de livros:', len(boosk_price_range))
print('Menor preço:', boosk_price_range['price'].min())
print('Maior preço:', boosk_price_range['price'].max())
print('Preço médio:', boosk_price_range['price'].mean())

,name,price,availability,rating
5,The Requiem Red,22.65,In stock,1
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,4
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,4
11,Shakespeare's Sonnets,20.66,In stock,4
14,Rip it Up and Start Again,35.02,In stock,5
16,Olio,23.88,In stock,1
17,Mesaerion: The Best Science Fiction Stories 18...,37.59,In stock,1
21,How Music Works,37.32,In stock,2
22,Foolproof Preserving: A Guide to Small Batch J...,30.52,In stock,3
23,Chase Me (Paris Nights #2),25.27,In stock,5


Total de livros: 401
Menor preço: 20.02
Maior preço: 39.77
Preço médio: 30.063067331670826


In [20]:
selected_books = df_all_products[
    df_all_products["rating"].isin([4, 5])
]

display(selected_books.head(10))
print('Total de livros selecionados:', len(selected_books))

,name,price,availability,rating
3,Sharp Objects,47.82,In stock,4
4,Sapiens: A Brief History of Humankind,54.23,In stock,5
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,4
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,4
11,Shakespeare's Sonnets,20.66,In stock,4
12,Set Me Free,17.46,In stock,5
13,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,In stock,5
14,Rip it Up and Start Again,35.02,In stock,5
23,Chase Me (Paris Nights #2),25.27,In stock,5
24,Black Dust,34.53,In stock,5


Total de livros selecionados: 375


In [21]:
filtered_book = df_all_products.query(
    'price >= 20 and price <= 40 and rating >=4'
)

display(filtered_book.head(10))
print('Total de livros filtrados:', len(filtered_book))


,name,price,availability,rating
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,4
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,4
11,Shakespeare's Sonnets,20.66,In stock,4
14,Rip it Up and Start Again,35.02,In stock,5
23,Chase Me (Paris Nights #2),25.27,In stock,5
24,Black Dust,34.53,In stock,5
32,The Elephant Tree,23.82,In stock,5
43,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,In stock,5
60,The Nameless City (The Nameless City #1),38.16,In stock,4
63,The Mindfulness and Acceptance Workbook for An...,23.89,In stock,4


Total de livros filtrados: 138


In [22]:
rating_count = df_all_products['rating'].value_counts()
display(rating_count.to_frame(name='book_count'))

,book_count
rating,
1,226
3,203
5,196
2,196
4,179


In [24]:
rating_sumary = df_all_products.groupby('rating')['price'].mean()

display(rating_sumary.to_frame(name='average_price'))

,average_price
rating,
1,34.561195
2,34.810918
3,34.692020
4,36.093296
5,35.374490


In [25]:
rating_statistics = df_all_products.groupby('rating').agg(
    book_count=('price', 'count'),
    average_price=('price', 'mean'),
    minimum_price=('price', 'min'),
    maximum_price=('price', 'max')
)
display(rating_statistics)



,book_count,average_price,minimum_price,maximum_price
rating,,,,
1,226,34.561195,10.40,59.64
2,196,34.810918,10.02,59.95
3,203,34.692020,10.16,59.99
4,179,36.093296,10.01,59.45
5,196,35.374490,10.00,59.92


In [26]:

rating_statistics = rating_statistics.reset_index()

display(rating_statistics)


,rating,book_count,average_price,minimum_price,maximum_price
0,1,226,34.561195,10.40,59.64
1,2,196,34.810918,10.02,59.95
2,3,203,34.692020,10.16,59.99
3,4,179,36.093296,10.01,59.45
4,5,196,35.374490,10.00,59.92


In [27]:
df_loads = pd.DataFrame({
    'load_id': [101, 102, 103, 104],
    'carrier_id': [1, 2, 1, 3],
    'volume_count':[120, 85, 200, 95]
})

df_carriers = pd.DataFrame({
    'carrier_id': [1, 2, 3],
    'carrier_name': ['Transportadora A', 'Transportadora B', 'Transportadora C']
})

display(df_loads)
display(df_carriers)

,load_id,carrier_id,volume_count
0,101,1,120
1,102,2,85
2,103,1,200
3,104,3,95


,carrier_id,carrier_name
0,1,Transportadora A
1,2,Transportadora B
2,3,Transportadora C


In [28]:
df_merged = pd.merge(
    df_loads,
    df_carriers,
    on='carrier_id', 
    how='left'
)

display(df_merged)


,load_id,carrier_id,volume_count,carrier_name
0,101,1,120,Transportadora A
1,102,2,85,Transportadora B
2,103,1,200,Transportadora A
3,104,3,95,Transportadora C


In [29]:
df_page1 = pd.DataFrame({
    'load_id': [101, 102],
    'volume_count': [120, 85]
})

df_page2 = pd.DataFrame({
    'load_id': [103, 104],
    'volume_count': [200, 95]
})

df_all_load =pd.concat(
    [df_page1, df_page2],
    ignore_index=True
)

display(df_all_load)

,load_id,volume_count
0,101,120
1,102,85
2,103,200
3,104,95
